# Build dataset (pas-à-pas)

Ce notebook reprend la logique de `src/build_dataset_from_csv.py` de façon pédagogique.

Objectif: comprendre et exécuter chaque étape avant de matérialiser `train/val/test` dans `data/out`.

## 1) Imports et accès au projet

On ajoute la racine du projet au `sys.path` pour importer les fonctions du script.

In [13]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if (project_root / 'src').exists():
    pass
elif (project_root.parent / 'src').exists():
    project_root = project_root.parent
else:
    raise RuntimeError('Impossible de trouver le dossier src depuis le notebook.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print('Project root:', project_root)

Project root: /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain


## 2) Charger la configuration et les fonctions du script

In [14]:
from src.utils import load_config, ensure_dir
from src.build_dataset_from_csv import (
    RANDOM_SEED,
    find_first_csv,
    infer_label_from_filename_parent,
    normalize_label_value,
    detect_images_root_from_filenames,
    materialize_split,
    clean_output_root,
)

cfg = load_config(project_root / 'config.yaml')
kaggle_root = project_root / cfg['paths']['kaggle_root']
out_root = project_root / cfg['paths']['keras_root']
images_hint = cfg['paths'].get('images_subdir_hint', 'images')

print('kaggle_root =', kaggle_root)
print('out_root    =', out_root)

kaggle_root = /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/data/in/kaggle-wikiart
out_root    = /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/data/out


## 3) Lire le CSV et inspecter les colonnes

In [15]:
csv_path = find_first_csv(kaggle_root)
df_raw = pd.read_csv(csv_path)

print('CSV:', csv_path)
print('Shape:', df_raw.shape)
print('Columns:', list(df_raw.columns))
df_raw.head(3)

CSV: /Users/lewagon/code/Lionel-Nokia/art-xplain/art-xplain/data/in/kaggle-wikiart/classes.csv
Shape: (80042, 9)
Columns: ['filename', 'artist', 'genre', 'description', 'phash', 'width', 'height', 'genre_count', 'subset']


,filename,artist,genre,description,phash,width,height,genre_count,subset
0,Abstract_Expressionism/aaron-siskind_acolman-1...,aaron siskind,['Abstract Expressionism'],acolman-1-1955,bebbeb018a7d80a8,1922,1382,1,train
1,Abstract_Expressionism/aaron-siskind_chicago-6...,aaron siskind,['Abstract Expressionism'],chicago-6-1961,d7d0781be51fc00e,1382,1746,1,train
2,Abstract_Expressionism/aaron-siskind_glouceste...,aaron siskind,['Abstract Expressionism'],gloucester-16a-1944,9f846e5a6c639325,1382,1857,1,train


## 4) Préparer `filename` + `label`

- Priorité: 

  - Le `label`est retrouvé depuis le parent de `filename` (`Style/image.jpg` -> `Style`).
  - Sinon (fallback) à partir d'une colonne (`style`, `movement`, `genre`, `artist`).

In [ ]:
# 1) Mapping des colonnes en minuscules pour éviter les problèmes de casse
cols = {c.lower(): c for c in df_raw.columns}

# 2) La colonne filename est obligatoire: elle contient le chemin relatif de l'image
if 'filename' not in cols:
    raise ValueError(f"Colonne 'filename' introuvable. Colonnes={list(df_raw.columns)}")

# 3) Colonne source image + copie de travail du DataFrame
filename_col = cols['filename']
df = df_raw.copy()

# 4) Colonne standardisée qui contiendra le label final
label_col = '__label__'

# 5) On essaie d'abord d'inférer le label depuis le parent du filename
#    Exemple: 'Impressionism/abc.jpg' -> 'Impressionism'
derived_labels = infer_label_from_filename_parent(df[filename_col])
if derived_labels is not None:
    df[label_col] = derived_labels
    print('Label source: parent folder du filename')
else:
    # 6) Fallback: on cherche une colonne de label explicite dans le CSV
    selected_label_col = None
    for cand in ['style', 'movement', 'genre', 'artist']:
        if cand in cols:
            selected_label_col = cols[cand]
            break
    if selected_label_col is None:
        raise ValueError('Aucune colonne label trouvée parmi: style/movement/genre/artist')
    df[label_col] = df[selected_label_col]
    print(f"Label source: colonne CSV '{selected_label_col}'")

# 7) On garde uniquement (filename, label), puis on nettoie les valeurs
#    normalize_label_value gère notamment les labels stockés comme listes en texte
df = df[[filename_col, label_col]].dropna().copy()
df[label_col] = df[label_col].astype(str).map(normalize_label_value)

df.head(3)

## 5) Détecter la racine des images et filtrer les styles

In [ ]:
# 1) Détecter automatiquement la racine des images à partir des chemins CSV
#    Exemple: si filename='Style/image.jpg', la racine peut être kaggle_root directement.
images_root = detect_images_root_from_filenames(kaggle_root, images_hint, df[filename_col])

# 2) Lire les hyperparamètres de filtrage depuis config.yaml
keep_top_styles = int(cfg['dataset']['keep_top_styles'])
min_images_per_style = int(cfg['dataset']['min_images_per_style'])

# 3) Compter les images par style, garder les classes assez représentées,
#    puis limiter aux N styles les plus fréquents
counts = df[label_col].value_counts()
keep = counts[counts >= min_images_per_style].head(keep_top_styles).index
df_filtered = df[df[label_col].isin(keep)].copy()

# 4) Afficher un résumé de ce qui sera utilisé pour le split
print('Images root:', images_root)
print('Styles gardés:', list(keep))
print('Total rows après filtrage:', len(df_filtered))
counts.head(10)


## 6) Split train / val / test 

In [ ]:
# 1) Charger les ratios de split depuis config.yaml
test_size = float(cfg['dataset']['test_size'])
val_size = float(cfg['dataset']['val_size'])

# 2) Premier split: train vs (val+test), stratifié par label
#    La stratification conserve la proportion des styles dans chaque sous-ensemble.
train_df, tmp_df = train_test_split(
    df_filtered,
    test_size=(test_size + val_size),
    random_state=RANDOM_SEED,
    stratify=df_filtered[label_col],
)

# 3) Deuxième split: séparation de tmp_df en val et test
rel_val = val_size / (test_size + val_size)
val_df, test_df = train_test_split(
    tmp_df,
    test_size=(1 - rel_val),
    random_state=RANDOM_SEED,
    stratify=tmp_df[label_col],
)

# 4) Vérification des volumes
print('train:', len(train_df), 'val:', len(val_df), 'test:', len(test_df))


## 7) Optionnel: nettoyer `out_root`

Mettre `DO_CLEAN = True` pour supprimer le contenu actuel de `data/out` avant régénération.

In [ ]:
# Mettre à True pour nettoyer complètement data/out avant de recopier les images
DO_CLEAN = True #False

# clean_output_root supprime TOUT ce qu'il y a dans out_root (dossiers et fichiers)
if DO_CLEAN:
    clean_output_root(out_root)
else:
    print('Nettoyage ignoré (DO_CLEAN=False)')


## 8) Matérialiser les splits (copie des images)

Mettre `RUN_MATERIALIZE = True` pour lancer la copie dans `data/out/{train,val,test}/<style>/...`.

- _Attendre car c'est long (entre 60s et 90s)..._

In [ ]:
# Mettre à True pour lancer la copie physique des fichiers dans data/out/{train,val,test}
RUN_MATERIALIZE = True #False

if RUN_MATERIALIZE:
    # 1) S'assurer que la structure de sortie existe
    ensure_dir(out_root / 'train')
    ensure_dir(out_root / 'val')
    ensure_dir(out_root / 'test')


    # 2) Pour chaque split, copier les images dans le bon dossier de style
    for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        copied, missing = materialize_split(
            split_df, split_name, out_root, images_root, filename_col, label_col
        )
        print(f"{split_name}: copied={copied} missing={missing} total_rows={len(split_df)}")
else:
    print('Materialization ignorée (RUN_MATERIALIZE=False)')

print("\n- Les missing sont les lignes pour lesquelles le script n’a pas trouvé le fichier image source")


## 9) Vérification rapide du résultat

In [ ]:
# Contrôle rapide: compter le nombre de styles et de fichiers JPG par split
for split in ['train', 'val', 'test']:
    split_dir = out_root / split
    if not split_dir.exists():
        print(split, 'absent')
        continue

    # Un style = un sous-dossier
    style_dirs = [p for p in split_dir.iterdir() if p.is_dir()]

    # Comptage simple des .jpg pour vérifier que la copie a bien eu lieu
    n_files = sum(1 for d in style_dirs for _ in d.rglob('*.jpg'))
    print(f"{split}: styles={len(style_dirs)} - images(s) jpg={n_files}")
